# 01b · A first look at ADS-B Exchange data over India (Phase 1, task 1.2)

Exploration only. Uses the small real slices ingested on the laptop:
`uv run stratum ingest-adsb --day 2026-01-01 --hours 6 --max-snapshots 60` (5 min) and
`--day 2026-02-01 --hours 0 --max-snapshots 120` (10 min). Checks: positions fall on known
air routes, cruise altitudes cluster at flight levels, reports arrive every ~5 s.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import shapely

from stratum.config import corridor_polygon, load_domain
from stratum.flights.clean import daily_summary
from stratum.io.adsb_adsbx import read_day
from stratum.paths import drive_root, repo_root

days = {d: read_day(drive_root() / "raw/adsb/adsbx" / d) for d in ["2026-01-01", "2026-02-01"]}
for d, df in days.items():
    print(d, {k: v for k, v in daily_summary(df).items() if k != "removed_in_basic_clean"})

In [ ]:
# Sanity plot: positions coloured by altitude, the corridor, and the altitude distribution.
dom = load_domain()
corr = corridor_polygon(dom.corridor)
df = days["2026-02-01"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw={"width_ratios": [1.3, 1]})
sc = ax1.scatter(df["lon"], df["lat"], c=df["baro_alt_ft"] / 100, s=0.5, cmap="cividis", vmin=200, vmax=420)
ax1.plot(*corr.exterior.xy, color="#c0392b", lw=1.2, label="DEL–BOM corridor (±75 km)")
b = dom.study_bbox
ax1.plot([b.lon[0], b.lon[1], b.lon[1], b.lon[0], b.lon[0]], [b.lat[0], b.lat[0], b.lat[1], b.lat[1], b.lat[0]], "k--", lw=0.7)
fig.colorbar(sc, ax=ax1, label="flight level")
ax1.set_title("ADS-B Exchange, 2026-02-01 00:00–00:10 UTC, ≥ FL200", fontsize=10)
ax1.set_xlabel("lon (°E)"); ax1.set_ylabel("lat (°N)"); ax1.legend(loc="lower left", fontsize=8)
ax2.hist(df["baro_alt_ft"] / 100, bins=np.arange(200, 460, 5), color="#2a4a8f")
ax2.set_xlabel("barometric flight level"); ax2.set_ylabel("position reports")
ax2.set_title("Cruise levels cluster at standard flight levels", fontsize=10)
out = repo_root() / "reports/phase1/adsb_sanity_2026-02-01T00.png"
fig.tight_layout(); fig.savefig(out, dpi=130)
inside = shapely.contains_xy(corr, df["lon"].to_numpy(), df["lat"].to_numpy())
print("aircraft in corridor ≥ FL250:", df[inside & (df["baro_alt_ft"] >= 25000)]["icao24"].nunique(), "| saved", out)